# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [3]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Thu Nov 19 19:18:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [4]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [5]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [6]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Is it possible to do something different in quantum computing? What is the difference between quantum computing and classical computing? Is there any difference in hardware? How does a quantum computing work? What are the different types of quantum computing? What are the quantum computing applications? What are the advantages of quantum computing? What are the disadvantages of quantum computing?
Quantum computing is a new approach of computation that uses quantum physics principles to solve complex problems.
Quantum computing is a new approach of computation that uses quantum physics principles to solve complex problems. It’s a way to use the power of the quantum world, which is a world that is very different from the world we live in. The quantum world is a world of particles and waves, where the rules are different

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a revolutionary field of computer science that leverages the principles of qu

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [8]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve 15 × 24, we can use the standard multiplication algorithm. Here's how we can break it down:

1. **Multiply 15 by 4 (the units digit of 24):**
   15 × 4 = 60

2. **Multiply 15 by 20 (the tens digit of 24, which is equivalent to 2 × 10):**
   15 × 20 = 15 × 2 × 10 = 30 × 10 = 300

3. **Add the two results together:**
   60 + 300 = 360

So, 15 × 24 = 360.

Alternatively, we can also use the distributive property of multiplication:
15 × 24 = 15 × (20 + 4) = 15 × 20 + 15 × 4 = 300 + 60 = 360.

Both methods give us the same result.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To solve this problem, we need to determine the amount of flour needed per cookie and then multiply that by the number of cookies we want to make.

First, let's find out how much flour is needed per cookie. Th

In [1]:
# Повторний запуск /think з більшим лімітом: max_new_tokens=1000 (замість 300).
# Середовище було перезапущене, тому завантажуємо лише instruct-модель і знову задаємо generate.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

instruct_model_name = "HuggingFaceTB/SmolLM3-3B"
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)

def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

print("===== Режим: /think, max_new_tokens=1000 =====\n")
for i, prompt in enumerate(reasoning_prompts, 1):
    messages = [
        {"role": "system", "content": "/think"},
        {"role": "user", "content": prompt},
    ]
    formatted = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=1000, temperature=0.3)
    print(f"Задача {i}: {prompt}")
    print(f"Відповідь: {answer}")
    print("\n" + "-" * 50 + "\n")

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

===== Режим: /think, max_new_tokens=1000 =====



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Задача 1: What is 15 × 24? Show your work.
Відповідь: <think>
Okay, so I need to figure out what 15 multiplied by 24 is. Hmm, let me think about how to approach this. I remember that multiplication can be done in a few different ways, like using the standard algorithm, breaking it down into smaller parts, or maybe even using mental math tricks. Let me try a couple of methods to see which one works best for me.

First, maybe I'll try the standard algorithm. That's where you multiply each digit of the second number by the first number and then add up the results. Let's see. So, 24 has two digits: 2 and 4. I'll start by multiplying 15 by 4, then multiply 15 by 20, and then add those two results together.

Starting with 15 times 4. Let me do that step by step. 15 times 4. 5 times 4 is 20, and then 1 times 4 is 4. So, 20 plus 4 is 24. Wait, no, that's not right. Wait, actually, when you multiply 15 by 4, you have to consider the place value. Since 15 is 10 + 5, multiplying by 4 would be (10

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [10]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [11]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [12]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [13]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [14]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [15]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## Відповіді на запитання 0.1 (питання 11–21)

**11. Як довго виконувалися клітинки ноутбуку 02_chat_templates? Чи були проблеми з запуском, які?**

Запуск через «Виконати всі» на GPU Tesla T4 (15,6 ГБ памʼяті) зайняв приблизно 7 хвилин. Найдовше — завантаження двох моделей SmolLM3-3B (Base й Instruct, по ~6 ГБ кожна): 4 хвилини. Порівняння відповідей base- та instruct-моделі — 13 с, шість генерацій у режимах /no_think і /think — 57 с, показ шаблонів різних моделей — 6 с, отримання списку частин SmolTalk2 — 1 с, завантаження частини датасету — 1 с, перетворення GSM8K — 2 с. Помилок не було. Виводились лише попередження бібліотеки transformers (про clean_up_tokenization_spaces для BPE-токенізатора та про поле rope_parameters у конфігурації Phi-3.5), які не впливають на результат. Дві моделі у float16 (≈6 ГБ + ≈6 ГБ) вміщуються в памʼять T4.

**12. Для чого змінювати параметр add_generation_prompt в apply_chat_template()? Як по згенерованій розмові дізнатися його значення?**

add_generation_prompt=False потрібен для завершених діалогів, тобто для тренувальних даних: рядок закінчується останнім повідомленням. add_generation_prompt=True додає в кінець заголовок відповіді асистента (`<|im_start|>`assistant), щоб модель знала, що тепер її черга генерувати відповідь; це використовується для інференсу. Визначити значення по рядку можна за його кінцівкою: якщо текст закінчується на `<|im_start|>`assistant (для SmolLM3 у режимі /no_think після нього ще порожній блок `<think></think>`), то було True; якщо закінчується на `<|im_end|>` останнього повідомлення (у виводі — на запитанні користувача), то було False.

**13. Чим відрізняються ролі system від assistant у шаблонах розмов?**

system — це інструкції розробника, які задають роль, стиль і обмеження моделі (у SmolLM3 там також прапорець /think чи /no_think); зазвичай він один і стоїть на початку розмови. Модель його не генерує, це лише контекст. assistant — це відповіді моделі: у попередніх ходах розмови вони є історією, під час інференсу їх генерує модель, а під час SFT саме на токенах assistant вона вчиться. Для кожної ролі шаблон ставить свою мітку (`<|im_start|>`system, `<|im_start|>`assistant), тому модель розрізняє, хто говорить.

**14. Прокоментуйте вивід у секції 2. Які секції додав шаблонізатор і для чого?**

Для чотирьох діалогів виведено завершений варіант (False) та варіант для інференсу (True). Шаблонізатор додав:

- блок «## Metadata»: Knowledge Cutoff Date (June 2025), Today Date (поточна дата — 19 November 2026, підставляється автоматично) і Reasoning Mode (/think або /no_think). Прапорець із нашого системного повідомлення перенесено в цей рядок і прибрано з тексту інструкцій;
- блок «## Custom Instructions» з нашим системним повідомленням. Якщо власної інструкції немає (діалог simple_qa), підставляється стандартна «You are a helpful AI assistant named SmolLM, trained by Hugging Face.». У режимі /think (діалог reasoning_task) підставляється довга стандартна інструкція про розділи Thought і Solution, яка змушує модель міркувати;
- службові токени `<|im_start|>`роль … `<|im_end|>` навколо кожного повідомлення (формат ChatML). У виводі системний блок не закривається `<|im_end|>`, а відділяється порожнім рядком від наступного повідомлення;
- при add_generation_prompt=True у кінці додано `<|im_start|>`assistant. У режимі /no_think після нього ще порожній блок `<think></think>`, щоб модель одразу відповідала без міркувань; у /think блоку немає, модель сама починає з `<think>`;
- у діалозі multi_turn до попередньої відповіді асистента теж додано порожній блок `<think></think>`, щоб історія відповідала режиму /no_think.

Тобто модель бачить значно більше, ніж ми написали, і саме так вона була навчена.

**15. Різниця між Base і Instruct (секція 3).**

Base-модель отримала сирий текст «Explain me, what is quantum computing?» і продовжила його як документ, а не відповіла на питання: згенерувала список нових питань про квантові обчислення, потім двічі повторила те саме речення («Quantum computing is a new approach of computation…») і почала розповідати загальні речі. Відповіді як такої немає, є повтори й відхилення від теми. Instruct-модель отримала діалог за шаблоном і дала зв'язне структуроване пояснення: що таке кубіти, суперпозиція, заплутаність, порівняння з класичними комп'ютерами, застосування (текст обірвано на 150 токенах через max_new_tokens=150). Base навчена лише передбачати наступний токен, а Instruct додатково донавчена (SFT) виконувати інструкції та вести діалог. Саме з base-моделі зазвичай починають власний SFT; шаблону в її токенізаторі немає (chat_template is not None → False).

**16. Чому в режимі /think при генерації передано max_new_tokens=300?**

У режимі `/think` модель спочатку пише довгі міркування в блоці `<think>…</think>` і лише потім відповідь, тому їй потрібно значно більше токенів, ніж у `/no_think`. Обмеження `max_new_tokens` потрібне, щоб генерація не тривала занадто довго й не займала зайву памʼять T4. Значення 300 виявилось замалим: у всіх трьох задачах відповідь обірвалась посеред міркувань і до фінальної відповіді модель не дійшла. Тому я повторно запустив окрему клітинку з `max_new_tokens=1000` (≈3 хв на T4): задачі 2 і 3 завершили міркування й дали остаточну відповідь (5 чашок; \$18.95), а в задачі 1 міркування знову не вклались у 1000 токенів (модель багато разів перевіряла результат 360 різними способами), тож остаточної відповіді там немає. Висновок: для `/think` потрібен великий ліміт (1000 токенів і більше), а для простих задач достатньо `/no_think`.

**17. Чи впоралася модель з математичними задачами? Який режим обрав би я?**

У режимі `/no_think` — так: 15 × 24 = 360 (двома способами), 5 чашок борошна для 30 печив, залишок \$18.95; усі три відповіді правильні й з поясненнями. У режимі `/think` з лімітом 300 токенів до відповіді модель не дійшла. З лімітом 1000 токенів: задача 2 — правильно (5 чашок), задача 3 — правильно (\$18.95, з перевіркою через центи), у задачі 1 міркування (проміжний результат 360 правильний) не завершилось в межах ліміту, тож остаточної відповіді немає. Для простих арифметичних задач я обрав би `/no_think`: швидше, дешевше за токенами і відповідь з кроками одразу (достатньо попросити «Show your work»), тоді як `/think` на таких задачах дає зайві повторні перевірки. `/think` має сенс для складніших багатокрокових задач із великим `max_new_tokens`; результат 3-мільярдної моделі все одно треба перевіряти.

**18. Які відмінності в шаблонах розмов помічено в секції 5?**

Усі моделі мають свій формат:

- SmolLM3-3B: ChatML (`<|im_start|>`/`<|im_end|>`), додатковий блок Metadata (дати, режим міркування) і Custom Instructions;
- Qwen3-0.6B: теж ChatML, але без метаданих — системне повідомлення йде як є; перед відповіддю асистента в історії вставляється порожній блок `<think></think>`;
- Phi-3.5-mini-instruct: інші службові токени — `<|system|>`, `<|user|>`, `<|assistant|>`, кінець повідомлення `<|end|>`, у кінці ще `<|endoftext|>`.

Режим міркування також задається по-різному: у SmolLM3 прапорцем /think чи /no_think у системному повідомленні, у Qwen3 параметром enable_thinking (True — рядок закінчується на `<|im_start|>`assistant; False — додається порожній `<think></think>`). Висновок: ролі system/user/assistant — це лише домовленість формату messages, а в токени кожна модель перетворює їх по-своєму, тому шаблон треба брати саме від тієї моделі, яку використовуємо.

**19. Для чого знати про шаблони розмов, якщо ми хочемо донавчити модель через SFT?**

Під час SFT модель має бачити дані точно в тому самому форматі, у якому потім працюватиме на інференсі. Якщо шаблон різний, модель вчиться на розмітці, якої не буде під час використання: плутає ролі, не завершує відповідь токеном `<|im_end|>`, ігнорує режими. У base-моделі шаблону немає, тому його треба задати самостійно (найпростіше взяти з instruct-версії), а нові службові токени для base-моделі раніше не мали осмисленого значення. Шаблон також додає метадані та режим міркування, які впливають на навчання (приклад з GSM8K: у даних немає міркувань, а шаблон ставить /think). Крім того, SFTTrainer сам застосовує шаблон до формату messages і від нього залежить, які токени вважати відповіддю асистента (див. assistant_only_loss у демо 3).

**20. З яких даних побудовано датасет HuggingFaceTB/smoltalk2?**

SmolTalk2 (конфігурація SFT) — це суміш з 25 частин (splits) з різних джерел; назви частин закінчуються на _think (відповіді з міркуваннями) або _no_think (без міркувань). До них входять: відкриті набори даних із міркуваннями та математикою/науками (OpenThoughts3-1.2M, s1k-1.1, Mixture_of_Thoughts_science, multi_turn_reasoning_if), загальні набори інструкцій (OpenHermes 2.5, Tulu 3 SFT personas instruction following, Aya dataset), дані для виклику функцій та агентів (hermes_function_calling_v1, smolagents_toolcalling_traces, xlam_traces), довгий контекст (LongAlign 64k), таблиці (table_gpt), а також синтетичні діалоги, створені для SmolLM3 (повсякденні розмови, переписування, summarize, Magpie Ultra, системні промпти, 8 мов), частина яких згенерована сильнішою моделлю Qwen3-32B. У демо ми використали частину smoltalk_everyday_convs_reasoning_Qwen3_32B_think (2057 діалогів зі стовпцями messages, chat_template_kwargs і source).

**21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат messages»?**

Беремо перші 100 прикладів GSM8K з колонками question і answer. Функція to_messages (через map з batched=True) для кожної пари створює діалог із трьох повідомлень: system («You are a math tutor. Solve problems step by step.» — додали ми, у даних його не було), user (текст питання) і assistant (текст розвʼязку без змін, разом з анотаціями `<<48/2=24>>` і фінальним рядком «#### 72»). Параметр remove_columns видаляє початкові колонки, тож лишається одна колонка messages — список словників {"role", "content"}. Далі шаблон розмови перетворює цей список на рядок для тренування (145 токенів для першого прикладу).